# C-test ONE · RT-Measure

*Ctest-RT-Measure-v1.5.3*

Types the least expensive description of each item into retrieval and
transformation steps, with an LLM judge asked five times at temperature 0.4, and records the
median counts $\hat r$ and $\hat t$ and their spread.

| reads | writes |
|---|---|
| `ctest_battery.json`, `ctest_pricing_<machine>.json` | $\hat r$ and $\hat t$ onto `ctest_battery.json` |

**Needs** a Gemini API key. No GPU. The machine whose descriptions are typed is chosen by
`PRICING`.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


## 1 · Install

In [ ]:
# ---------------------------------------------------------------------------
# If the key cannot be found in the secrets panel, put it here. The environment
# is checked before any provider, so this always wins.
#
# Kaggle secrets are PER NOTEBOOK: having one in another session does not carry
# over. Add-ons -> Secrets, and switch the toggle on for this notebook.
API_KEY_HERE = ""       # e.g. "AIza..."  Leave empty to use the secrets panel.
# ---------------------------------------------------------------------------
import os
if API_KEY_HERE:
    os.environ["GOOGLE_API_KEY"] = API_KEY_HERE
    print("GOOGLE_API_KEY taken from API_KEY_HERE in this cell")

%pip install -q -U google-genai

import os
try:
    import google.colab
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as _e:
    print("not on Colab, or Drive not mounted:", _e)


## 1b · Preamble

In [ ]:
import os, sys, re, json, time, glob, math, shutil, random, hashlib, platform
import collections, statistics as st, datetime, threading
from concurrent.futures import ThreadPoolExecutor, as_completed

BUILD = "Ctest-RT-Measure-v1.5.3"
print("=" * 72)
print(f"{BUILD}")
print("=" * 72)

# The filesystem decides, not an import. Kaggle ships packages that make
# `import google.colab` succeed, which sent ENV to "colab" on Kaggle and killed
# the run on the unmounted-Drive guard.
if os.path.exists("/kaggle"):
    ENV = "kaggle"
elif os.path.isdir("/content"):
    ENV = "colab"
else:
    ENV = "local"
# Two separate places, because they are separate things and always were.
DRIVE_FOLDER   = "ctest"        # Colab: /content/drive/MyDrive/ctest
KAGGLE_DATASET = "ctest-one-data"    # Kaggle: the dataset to read from and push to.
                                # A Kaggle dataset is a FOLDER of files, not a
                                # single file.

DRIVE = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
# OUTDIR is a second place to copy finished files to. On Colab that is Drive.
# On Kaggle there is no such place: an attached dataset is READ-ONLY at
# /kaggle/input, and the working directory is the only writable one and is
# already where files are written. Copying a file onto itself raises
# SameFileError, so OUTDIR stays "." and persistence is the dataset push in the
# cell below.
if ENV == "colab" and os.path.isdir("/content/drive/MyDrive"):
    OUTDIR = DRIVE
else:
    OUTDIR = "."
print(f"environment      {ENV}   python {platform.python_version()}")

# shutil.copy writes a FILE when its destination is not an existing directory, so
# without this every save lands in a file named after the folder and the results
# leave with the session.
if OUTDIR != ".":
    if os.path.exists(OUTDIR) and not os.path.isdir(OUTDIR):
        raise SystemExit(f"{OUTDIR} exists but is a file. Rename or delete it.")
    os.makedirs(OUTDIR, exist_ok=True)
elif ENV == "colab":
    raise SystemExit("Drive is not mounted, so anything written would be lost. Run\n"
                     "    from google.colab import drive; drive.mount('/content/drive')")
if ENV == "kaggle":
    print(f"outputs go to    the working directory, then pushed to the "
          f"{KAGGLE_DATASET} dataset")
    print( "                 (a dataset cannot be written to in place; see the "
           "persistence cell)")
if ENV != "kaggle":
    print(f"outputs go to    {OUTDIR}")

try:
    import torch
    if torch.cuda.is_available():
        print("GPU              attached, and not needed. Switch to CPU and save quota.")
except Exception:
    pass

def _secret(n):
    """Look for a secret, and SAY what each provider reported.

    An earlier version swallowed the Kaggle error and fell through to the Colab
    branch, which raises on Kaggle, so the result was a bare None and nothing
    said why. Kaggle secrets are PER NOTEBOOK: one attached to another session
    does not carry over."""
    if os.environ.get(n):
        return os.environ[n]
    notes = []
    try:
        from kaggle_secrets import UserSecretsClient
        v = UserSecretsClient().get_secret(n)
        if v:
            return v
        notes.append("kaggle: returned nothing")
    except ImportError:
        pass                                  # not on Kaggle, which is fine
    except Exception as e:
        notes.append(f"kaggle: {type(e).__name__}: {str(e)[:90]}")
    try:
        from google.colab import userdata
        v = userdata.get(n)
        if v:
            return v
        notes.append("colab: returned nothing")
    except ImportError:
        pass
    except Exception as e:
        notes.append(f"colab: {type(e).__name__}: {str(e)[:90]}")
    for m in notes:
        print(f"                 {m}")
    return None

_key = _secret("GOOGLE_API_KEY")
if _key: os.environ["GOOGLE_API_KEY"] = _key
print(f"GOOGLE_API_KEY   {'found' if _key else 'MISSING'}")

# The dataset, from a secret when there is one, so two accounts can work in
# parallel on separate copies with this notebook byte-identical in both.
_ds = _secret("KAGGLE_DATASET")
if _ds:
    KAGGLE_DATASET = _ds.strip()
print(f"KAGGLE_DATASET   {KAGGLE_DATASET}"
      + ("   from the secret" if _ds else "   the built-in default"))
if not _key:
    print()
    print("=" * 72)
    print("STOPPING: no GOOGLE_API_KEY, and the composer is an API model")
    print("=" * 72)
    if ENV == "kaggle":
        print("  Kaggle secrets are PER NOTEBOOK. Having one in another session")
        print("  does not carry over: Add-ons -> Secrets, and switch the toggle")
        print("  ON for this notebook. Any lines above say what the lookup")
        print("  reported.")
    elif ENV == "colab":
        print("  Colab: the key panel on the left, and the notebook-access")
        print("  toggle has to be on for this notebook.")
    print()
    print("  Or set API_KEY_HERE in the install cell, which is checked first.")
    raise SystemExit("no GOOGLE_API_KEY")

_PRINT_LOCK = threading.Lock()

def log(*a, **kw):
    with _PRINT_LOCK:
        print(*a, flush=True, **kw)

def stage(t):
    with _PRINT_LOCK:
        print(f"\n{'='*72}\n== {t}\n{'='*72}", flush=True)

def find(f):
    for p in [f, f"/content/{f}", f"{DRIVE}/{f}"] + glob.glob(f"/kaggle/input/**/{f}", recursive=True):
        if os.path.exists(p): return p
    return None

def load(name, default):
    p = find(name)
    if not p:
        log(f"  {name}: absent, starting empty"); return default
    log(f"  {name}: loaded from {p}")
    return json.load(open(p))


## 2 · The run

In [ ]:
# ============================== THE RUN ====================================
# r and t for every priced item, from p_star, the program that achieves K.
#
# Reads the battery and the pricing file, writes r_hat and t_hat back to the
# battery, and pushes. Touches nothing else.

# ---- carried from the generator, because the shared composer cell
# ---- uses them. Same values, so a call costs and retries the same.
COMPOSER     = "gemini-3.6-flash"
MAX_WORKERS  = 6
DISABLE_AFC = True
TOKEN_BUDGET, MAX_SPEND_USD = 150_000_000, 120.0
USD_PER_MTOK_IN, USD_PER_MTOK_OUT = 0.30, 3.75
API_RETRIES, API_BACKOFF = 3, 5
API_TIMEOUT_S      = 900
GEN_MAX_OUT, PROG_MAX_OUT, OTHER_MAX_OUT = 12000, 10000, 8000

BATTERY = "ctest_battery.json"
PRICING      = "ctest_pricing_qwen3-14b.json"   # the machine whose descriptions are typed:
                                               # _qwen3-8b or _phi-4-reasoning for the others
PRICE_TAG    = PRICING.replace("ctest_pricing_", "").replace(".json", "")

MEASURE_REPS = 5
MEASURE_TEMP = 0.4    # NOT 0. Three calls at 0 give three identical answers
MEASURE_TRIES = 9
MEASURE_ALL  = False  # False measures only what THIS machine has no reading
                      # for, which after a full run is the items added since.
                      # True redoes the battery, about 25 minutes.
MEASURE_ONLY = []     # named items only. Empty falls back to MEASURE_ALL.
MAX_OUT      = 8000

print(f"\nbattery          {BATTERY}")
print(f"pricing          {PRICING}")
print(f"typings          {MEASURE_REPS} each at temperature {MEASURE_TEMP}")
print(f"measures         {'every item' if MEASURE_ALL else 'only items with no r_hat'}")
print(f"reads            p_star, the description that achieved K, so that K,")
print(f"                 r_hat and t_hat all describe the same thing; then the")
print(f"                 bare, then the seed. Nothing is skipped.")
print(f"leaves alone     measured_r and measured_t, the admission-time values")


## 1c · Persistent state on Kaggle

In [ ]:
import subprocess

STATE_PATTERNS = ("ctest_*.json", "ctest_*.csv")

STATE_DATASET = ""        # "owner/slug"; empty derives it from KAGGLE_DATASET
STATE_PUSH_EVERY = 5      # 0 pushes only at the end of a run. On a long
                          # unattended run that is a gamble: save_all() keeps
                          # the working directory current, but the DATASET is
                          # only updated by a push, so a session cut before the
                          # end leaves the accumulated items unreachable from
                          # the next run.

def _kaggle_ready():
    if ENV != "kaggle":
        return False
    missing = []
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        v = _secret(k)
        if v:
            os.environ[k] = v
        else:
            missing.append(k)
    if missing:
        log(f"  state persistence OFF, missing secret(s): {', '.join(missing)}")
        log("     Add-ons -> Secrets -> add them from your kaggle.json, and turn")
        log("     the toggle on for this notebook. Results still appear in the")
        log("     version output; only state across versions is lost.")
        return False
    return True

KAGGLE_OK = _kaggle_ready()

def _slug():
    if STATE_DATASET:
        return STATE_DATASET
    return f"{os.environ.get('KAGGLE_USERNAME','unknown')}/{KAGGLE_DATASET.lower()}"

def state_files():
    """Every file the push should carry, newest copy of each name.

    Local first, since this run may have just written it. Then whatever is in
    the attached dataset and not local, so another notebook's output survives a
    push from this one."""
    chosen, source = {}, {}
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(pat)):
            chosen[os.path.basename(f)] = f
            source[os.path.basename(f)] = "this run"
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(f"/kaggle/input/**/{pat}", recursive=True)):
            b = os.path.basename(f)
            if b not in chosen:
                chosen[b] = f
                source[b] = "carried forward"
    return chosen, source

FRESH_DIR = "/kaggle/working/_fresh"

def _dataset_names():
    """Filenames actually IN the dataset, from the refreshed download.

    NOT from /kaggle/input. The mount is fixed when the session starts, so it
    can be older than the dataset, and comparing the mount against a push
    assembled from the mount compares a thing with itself and can never fire.
    That was the whole failure: the dataset held ctest_metrics.json, the mount
    did not, and a push assembled from the mount deleted it.

    If the refresh did not run or failed, this returns nothing and the guard
    stays quiet, because there is no trustworthy list to compare against.
    """
    out = set()
    if not os.path.isdir(FRESH_DIR):
        return out
    for pat in STATE_PATTERNS:
        for f in glob.glob(os.path.join(FRESH_DIR, pat)):
            out.add(os.path.basename(f))
    return out

STATE_REFRESH = True

def state_refresh():
    """Download the current dataset version into the working directory.

    Cheap, a couple of megabytes, and it removes a whole class of confusion:
    after this the notebook is reading what is IN the dataset rather than what
    happened to be mounted.

    Never raises. A failure leaves /kaggle/input in place, which is the old
    behaviour, and says so.
    """
    if not (STATE_REFRESH and KAGGLE_OK):
        return
    slug = _slug()
    d = FRESH_DIR
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    try:
        r = subprocess.run(["kaggle", "datasets", "download", "-d", slug,
                            "-p", d, "--unzip"],
                           capture_output=True, text=True, timeout=300)
        if r.returncode != 0:
            log(f"  could not refresh from {slug}: "
                f"{(r.stderr or r.stdout).strip()[:120]}")
            log(f"  falling back to whatever the session mounted, which may be "
                f"older than the dataset.")
            return
    except Exception as e:
        log(f"  could not refresh from {slug}: {str(e)[:100]}")
        return

    got, stale = [], []
    for pat in STATE_PATTERNS:
        for f in sorted(glob.glob(os.path.join(d, pat))):
            b = os.path.basename(f)
            mounted = None
            for p in glob.glob(f"/kaggle/input/**/{b}", recursive=True):
                mounted = p; break
            if os.path.exists(b):
                continue
            shutil.copy(f, b)
            got.append(b)
            if mounted is None:
                stale.append(f"{b} (NOT mounted at all)")
            elif os.path.getsize(mounted) != os.path.getsize(f):
                stale.append(f"{b} (mounted copy is a different size)")
    log(f"  refreshed {len(got)} file(s) from {slug}: {', '.join(sorted(got))}")
    if stale:
        log(f"  the session's mount was OUT OF DATE for: {', '.join(stale)}")
        log(f"  That is why this matters: without the refresh those would have "
            f"been read stale or not at all.")
def state_save(msg="update"):
    """Push a new version of the state dataset. Never raises."""
    if not KAGGLE_OK:
        return
    chosen, source = state_files()
    if not chosen:
        log("  no state files to push yet"); return
    d = "/kaggle/working/_state"
    shutil.rmtree(d, ignore_errors=True); os.makedirs(d, exist_ok=True)
    for b, f in chosen.items():
        shutil.copy(f, os.path.join(d, b))
    _in_dataset = _dataset_names()
    if not _in_dataset:
        log(f"  note: no refreshed copy of the dataset, so the push cannot be "
            f"checked for deletions. Set STATE_REFRESH = True.")
    _would_lose = _in_dataset - set(chosen)
    if _would_lose:
        log(f"  REFUSING TO PUSH. These are in the dataset and not in this "
            f"push, so the push would DELETE them:")
        for b in sorted(_would_lose):
            log(f"      {b}")
        log(f"  Most likely the session mounted an old version of the dataset. "
            f"STATE_REFRESH pulls the current one at startup; if that failed,")
        log(f"  the log above says why. Nothing has been pushed and nothing is "
            f"lost.")
        return

    carried = [b for b, s in source.items() if s == "carried forward"]
    log(f"  staging {len(chosen)} files"
        + (f", {len(carried)} carried forward from the dataset: "
           f"{', '.join(sorted(carried)[:4])}"
           f"{' ...' if len(carried) > 4 else ''}" if carried else ""))
    slug = _slug()
    json.dump({"title": slug.split("/")[-1], "id": slug,
               "licenses": [{"name": "CC0-1.0"}]},
              open(f"{d}/dataset-metadata.json", "w"))
    try:
        r = subprocess.run(["kaggle", "datasets", "version", "-p", d, "-m", msg,
                            "-r", "zip", "--dir-mode", "zip"],
                           capture_output=True, text=True, timeout=900)
        out = (r.stdout + r.stderr).strip()
        last = out.split("\n")[-1][:160] if out else ""
        if r.returncode == 0 and "error" not in out.lower():
            log(f"  state push: {last}")
        else:
            log(f"  the version push did not go through. Its output was:")
            for _l in out.split("\n")[-6:]:
                if _l.strip(): log(f"    {_l.strip()[:150]}")
            # Only create when the dataset really is absent. "Already in use"
            # means it exists and the version push failed for some other
            # reason, so creating it cannot help.
            missing = ("404" in out or "not found" in out.lower()) and \
                      "already in use" not in out.lower()
            if missing:
                r2 = subprocess.run(["kaggle", "datasets", "create", "-p", d,
                                     "-r", "zip", "--dir-mode", "zip"],
                                    capture_output=True, text=True, timeout=900)
                o2 = (r2.stdout + r2.stderr).strip()
                if "already in use" in o2.lower():
                    log(f"  the dataset exists after all, so the version push "
                        f"failed for another reason. NOTHING WAS PUSHED: "
                        f"download the files from this version's output.")
                else:
                    log(f"  created dataset '{slug}'. Attach it as an input so "
                        f"the next version can read it back.")
                    log(f"  state push: {o2.split(chr(10))[-1][:160]}")
            else:
                log(f"  NOTHING WAS PUSHED. The files are in this version's "
                    f"output and can be downloaded from there.")
    except Exception as e:
        log(f"  state push failed ({str(e)[:80]}). The results are still in this"
            f" version's output.")

stage("PERSISTENT STATE")
if ENV != "kaggle":
    log(f"  not on Kaggle, so nothing to do. Outputs go to {OUTDIR}.")
elif KAGGLE_OK:
    subprocess.run(["pip", "install", "-q", "kaggle"], capture_output=True)
    log(f"  credentials found for '{os.environ['KAGGLE_USERNAME']}'")
    log(f"  state dataset: {_slug()}")
    log(f"  pushes" + (f" every {STATE_PUSH_EVERY} items and" if STATE_PUSH_EVERY
        else "") + " at the end of the run")
_attached = sorted({os.path.basename(os.path.dirname(f))
                    for f in glob.glob("/kaggle/input/*/*")})
if ENV == "kaggle":
    if _attached:
        log(f"  datasets attached: {', '.join(_attached)}")
    else:
        log(f"  NO dataset attached. Add Input -> Datasets -> {KAGGLE_DATASET},")
        log(f"  or the loaders will not find the battery.")
_ch, _src = state_files()
if _ch:
    log(f"  state would carry {len(_ch)} files: {', '.join(sorted(_ch)[:6])}"
        + (" ..." if len(_ch) > 6 else ""))

state_refresh()


## 1d · What the run cost

In [ ]:
import csv, resource

RUNS_CSV = "ctest_runs.csv"

# Seed the local copy from wherever the file already lives, before anything is
# appended. Every other file is read back through find(), which globs
# /kaggle/input; this one was only ever appended, so on Kaggle it started empty
# each session and each run wrote a fresh one-row file. Worse, the dataset push
# prefers the local copy, so that one row REPLACED the accumulated history on
# every push. Rows were being lost, not merely not added.
_prev = find(RUNS_CSV)
if _prev and os.path.abspath(_prev) != os.path.abspath(RUNS_CSV):
    shutil.copy(_prev, RUNS_CSV)
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) carried in from {_prev}")
elif _prev:
    with open(RUNS_CSV) as _f:
        _n = max(0, sum(1 for _ in _f) - 1)
    print(f"run history       {_n} earlier run(s) already here")
else:
    print(f"run history       none yet, {RUNS_CSV} will be created")

_T_START = time.time()
_RUN_KEY = datetime.datetime.now().isoformat(timespec="seconds")
_UNITS_SO_FAR = [0]        # work finished, updated as the run goes
_RUN_NOTE = [""]           # what the row should say
_CPU_START = resource.getrusage(resource.RUSAGE_SELF).ru_utime + \
             resource.getrusage(resource.RUSAGE_SELF).ru_stime
_GPU_SECONDS = [0.0]      # wall clock with a model loaded; add to it when one is

def _gpu_name():
    try:
        import torch
        if torch.cuda.is_available():
            return torch.cuda.get_device_name(0)
    except Exception:
        pass
    return ""

def account(units=0, note=""):
    """Record this run's cost. Never raises: accounting must not stop work.

    UPSERT, not append. The row is keyed on when the run STARTED, so calling
    this repeatedly updates one row instead of adding several, and the numbers
    only grow. It used to be called once at the end, so a run stopped by hand
    or by the session ending wrote nothing at all and its compute vanished from
    the record.
    """
    try:
        cpu = (resource.getrusage(resource.RUSAGE_SELF).ru_utime +
               resource.getrusage(resource.RUSAGE_SELF).ru_stime) - _CPU_START
        tk = dict(TOKENS) if "TOKENS" in globals() else {}
        row = dict(
            run_started=_RUN_KEY,
            run_ended=datetime.datetime.now().isoformat(timespec="seconds"),
            build=BUILD, env=ENV, note=note,
            wall_s=round(time.time() - _T_START, 1),
            cpu_s=round(cpu, 1),
            gpu_s=round(_GPU_SECONDS[0], 1),
            gpu=_gpu_name(),
            units=units,
            model=(COMPOSER if "COMPOSER" in globals()
                   else REF_MACHINE if "REF_MACHINE" in globals() else ""),
            api_calls=tk.get("calls", 0),
            prompt_tokens=tk.get("in", 0),
            answer_tokens=tk.get("answer", 0),
            thinking_tokens=tk.get("think", 0),
            output_tokens=tk.get("out", 0),
            forward_passes=(REF.calls if "REF" in globals()
                            and hasattr(REF, "calls") else 0),
        )
        old_rows, old_cols = [], []
        if os.path.exists(RUNS_CSV):
            with open(RUNS_CSV, newline="") as f:
                rd = csv.DictReader(f)
                old_cols = list(rd.fieldnames or [])
                old_rows = list(rd)
        old_rows = [r for r in old_rows
                    if r.get("run_started") != _RUN_KEY]
        cols = old_cols + [c for c in row if c not in old_cols]
        with open(RUNS_CSV, "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=cols, restval="")
            w.writeheader()
            for r in old_rows + [row]:
                w.writerow({c: r.get(c, "") for c in cols})
        if OUTDIR != ".":
            shutil.copy(RUNS_CSV, OUTDIR)
    except Exception as e:
        log(f"  accounting failed ({str(e)[:70]}), which changes nothing else")


## 3 · The composer

In [ ]:
import logging

from google import genai

CLIENT = genai.Client()

class _AFCCount(logging.Handler):
    def __init__(self):
        super().__init__(); self.n = 0
    def emit(self, record):
        if "automatic function calling" in record.getMessage().lower():
            self.n += 1

_AFC = _AFCCount()

logging.getLogger("google_genai.models").addHandler(_AFC)

TOKENS = {"in": 0, "out": 0, "calls": 0, "empty": 0, "errors": 0, "truncated": 0,
          "think": 0, "answer": 0, "near_ceiling": 0, "tool_tokens": 0}

_TOK_LOCK, _SAVE_LOCK = threading.Lock(), threading.Lock()

def spend_usd():
    with _TOK_LOCK:
        return (TOKENS["in"]*USD_PER_MTOK_IN + TOKENS["out"]*USD_PER_MTOK_OUT)/1e6

def _text_of(r):
    """Thinking models often return the answer in parts rather than in .text."""
    if getattr(r, "text", None): return r.text
    out = []
    for c in (getattr(r, "candidates", None) or []):
        for p in (getattr(getattr(c, "content", None), "parts", None) or []):
            t = getattr(p, "text", None)
            if t and not getattr(p, "thought", False): out.append(t)
    return "".join(out)

def _finish(r):
    for c in (getattr(r, "candidates", None) or []):
        fr = getattr(c, "finish_reason", None)
        if fr is not None: return str(getattr(fr, "name", fr))
    return ""

def ask(prompt, temperature=1.0, max_out=None, timeout=None,
        model=None):
    """One metered call. Returns (text, note); note explains an unusual stop.

    `model` defaults to COMPOSER. The evaluator passes a different one per
    examinee, so this cell can stay identical across the notebooks rather than
    each keeping its own copy of the retry and metering logic.
    """
    model = model or COMPOSER
    max_out = max_out or OTHER_MAX_OUT
    if spend_usd() >= MAX_SPEND_USD: return None, "spend cap reached"
    with _TOK_LOCK:
        if TOKENS["in"] + TOKENS["out"] >= TOKEN_BUDGET:
            return None, "token budget reached"
    r = None
    for attempt in range(API_RETRIES):
        try:
            cfg = {"max_output_tokens": max_out, "temperature": temperature,
                   "http_options": {"timeout": int((timeout or API_TIMEOUT_S)*1000)}}
            if DISABLE_AFC:
                cfg["automatic_function_calling"] = {"disable": True}
            r = None
            for drop in ([], ["automatic_function_calling"],
                         ["automatic_function_calling", "http_options"]):
                try:
                    r = CLIENT.models.generate_content(
                        model=model, contents=prompt,
                        config={k: v for k, v in cfg.items() if k not in drop})
                    break
                except TypeError:
                    continue
            if r is None:
                raise TypeError("no accepted config shape")
            break
        except Exception as e:
            m = str(e).lower()
            transient = any(w in m for w in ("unavailable","overloaded","timeout",
                "deadline","timed out","rate limit","resource exhausted",
                "408","429","500","502","503","504"))
            with _TOK_LOCK: TOKENS["errors"] += 1
            if attempt == API_RETRIES-1 or not transient:
                return None, f"api error: {str(e)[:120]}"
            time.sleep(API_BACKOFF * (2**attempt))
    if r is None: return None, "no response"
    u, fr = getattr(r, "usage_metadata", None), _finish(r)
    with _TOK_LOCK:
        TOKENS["calls"] += 1
        if u:
            # Thinking is charged against max_output_tokens and is usually most
            # of it: one 9-digit multiplication cost 1,916 thinking tokens
            # against 18 of answer. That is why a call truncates with nothing
            # usable returned, and it is invisible unless counted apart.
            _think = getattr(u, "thoughts_token_count", 0) or 0
            _ans   = getattr(u, "candidates_token_count", 0) or 0
            TOKENS["in"]  += getattr(u, "prompt_token_count", 0) or 0
            TOKENS["out"] += _ans + _think
            TOKENS["think"] += _think
            TOKENS["answer"] += _ans
            if max_out and (_think + _ans) > 0.85 * max_out:
                TOKENS["near_ceiling"] += 1
            if getattr(u, "tool_use_prompt_token_count", None):
                TOKENS["tool_tokens"] += u.tool_use_prompt_token_count
        if fr and fr not in ("STOP", "FINISH_REASON_STOP"): TOKENS["truncated"] += 1
    t = _text_of(r)
    if not t:
        with _TOK_LOCK: TOKENS["empty"] += 1
        return None, f"empty reply (finish reason {fr or 'unknown'})"
    return t, ("" if fr in ("STOP","FINISH_REASON_STOP","") else f"finish reason {fr}")

def parse_json(text):
    """Close a truncated object before parsing, since the output budget covers
    the thinking too. Returns (obj, head) - head says why it failed.

    The closing is worked out from what is open rather than tried from a short
    list of endings. The list held four, and a reply cut deep inside a nested
    object needs more than any of them: v17-0081 failed every typing call in
    three separate runs for that reason, and the fragment was long and
    perfectly well formed up to the cut.
    """
    if not text: return None, ""
    t = re.sub(r"```(?:json)?", "", text)
    i = t.find("{")
    if i < 0: return None, "no opening brace. reply begins: " + t.strip()[:250]
    frag = t[i:]
    try: return json.loads(frag), ""
    except Exception: pass

    # walk it, tracking strings and escapes, and remember where the last
    # COMPLETE element ended: cutting back to there loses one entry and keeps
    # every other, which is better than losing the item.
    stack, in_str, esc, last_ok = [], False, False, None
    for k, ch in enumerate(frag):
        if esc: esc = False; continue
        if ch == "\\" and in_str: esc = True; continue
        if ch == '"': in_str = not in_str; continue
        if in_str: continue
        if ch in "{[": stack.append(ch)
        elif ch in "}]":
            if stack: stack.pop()
        if not in_str and ch in "},]" and len(stack) <= 2:
            last_ok = k

    for cand in (
            # close what is open, innermost first
            frag + ('"' if in_str else "") +
            "".join("}" if c == "{" else "]" for c in reversed(stack)),
            # or cut back to the last complete element and close from there
            (frag[:last_ok + 1] if last_ok else "") +
            "".join("}" if c == "{" else "]" for c in reversed(stack))):
        if not cand: continue
        try: return json.loads(cand), ""
        except Exception: pass
    return None, ("unparseable json, " + str(len(frag)) + " chars, "
                  + ("cut inside a string, " if in_str else "")
                  + str(len(stack)) + " unclosed. begins: " + frag[:200])

def tokens_report():
    with _TOK_LOCK: tk = dict(TOKENS)
    extra = "".join([f", {tk['errors']} api errors" if tk["errors"] else "",
                     f", {tk['empty']} empty" if tk["empty"] else "",
                     f", {tk['truncated']} truncated" if tk["truncated"] else "",
                     f", {tk['near_ceiling']} near the output ceiling"
                     if tk["near_ceiling"] else ""])
    share = tk["think"] / max(1, tk["think"] + tk["answer"])
    log(f"  composer: {tk['calls']} calls, {tk['in']+tk['out']:,} tokens "
        f"(~${spend_usd():.2f}){extra}")
    if tk["think"]:
        log(f"            thinking {tk['think']:,} of {tk['think']+tk['answer']:,} "
            f"output tokens ({share:.0%}); the model has no interpreter, so a long "
            f"sequence is simulated step by step")
    if tk["tool_tokens"]:
        log(f"            {tk['tool_tokens']:,} TOOL tokens: something IS calling "
            f"a tool, which changes what these runs mean")

def parallel(fn, jobs, workers=None, label=""):
    """Run fn over jobs. Results follow jobs, not completion order."""
    workers = workers or MAX_WORKERS
    if workers <= 1 or len(jobs) <= 1:
        return [fn(j) for j in jobs]
    out = [None] * len(jobs)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futs = {ex.submit(fn, j): i for i, j in enumerate(jobs)}
        done = 0
        for f in as_completed(futs):
            i = futs[f]
            try: out[i] = f.result()
            except Exception as e:
                out[i] = None
                log(f"      worker failed on job {i}: {str(e)[:110]}")
            done += 1
            if label and done % max(1, len(jobs)//10) == 0:
                log(f"      {label}: {done}/{len(jobs)}")
    return out

stage("LOADING")

_renamed = 0

if _renamed:
    log(f"  renamed `rule` to `seed` on {_renamed} items from an earlier battery")

EXCLUSIONS = ("duplicate", "quota", "flawed", "stale", "questionable")

_STOP = {"the", "each", "term", "terms", "subsequent", "starting", "start",
         "with", "next", "and", "for", "from", "list", "listed", "order",
         "their", "that", "this", "into", "then", "given", "using", "based"}

def _overlap(a, b):
    return len(a & b) / len(a | b) if (a and b) else 0.0

## 4 · The prompt

Seven worked examples, each a real rule with a settled answer.

In [ ]:
MEASURE = """Parse this instruction into components. Use exactly these types:

- "retrieval": anything that must be REMEMBERED rather than worked out. Naming
  a memorised set or ordering ("the western zodiac", "US states by admission",
  "the opening words of Moby Dick") AND reading a stored attribute off its
  members ("release year of", "atomic number of"). If answering needs something
  learnt, it is retrieval, even when phrased as an action.
  EXCEPT the free orderings, which are NOT retrieval and are "control":
  the counting numbers, the letters of the alphabet in order, and the digits.
  Everyone has these, so naming one retrieves nothing.
- "operation": anything DONE to the sequence or its terms that needs nothing
  remembered. Functions of a term ("square", "count the letters of", "first
  letter of") and rearrangements ("sorted alphabetically", "reversed", "every
  second one"). If it changes which terms appear, in what order, or what each
  becomes, it is an operation.
  EXCEPT a plain JOIN between the sources the rule already draws on, which is
  "control": running one after another, or alternating between them in their
  natural order. That is the least that having two sources requires.
- "control": the scaffolding that introduces or joins the output: "list", "for
  each", "then", "in order" when that means nothing more than the natural
  order. Also the two exemptions above.

QUOTE EACH THING ONCE. A rule that names the weekdays and then refers to them
three more times draws on ONE retrieval, not four. Count the distinct things
that must be remembered, not how often the rule mentions them.

Seven worked examples. These are real rules and the answers are settled.

  "1, 2, 3, 4 and so on"
    retrieval: none. Counting is free.        operation: none.
    -> r 0, t 0

  "the chemical elements in atomic order, by name"
    retrieval: the elements in atomic order   ONE
    operation: none
    -> r 1, t 0

  "Cumulative sum of the repeating sequence 3, 5, 5"
    retrieval: none. The three numbers are IN the rule, so nothing is
               remembered. A list written out is not a retrieval.
    operation: cycling the pattern; running total     TWO
    -> r 0, t 2

  "n -> first 3 chars of (reverse of nth color in cycling list: white, yellow,
   orange, green, blue, purple, brown, red, black) Caesar-shifted by n"
    retrieval: none. The nine colours are spelled out, so the reader remembers
               nothing. A rule that CONTAINS its list has r = 0.
    operation: cycle; reverse; first 3 chars; Caesar shift    FOUR
    -> r 0, t 4

  "term n = (white, yellow, green, blue, red, black)[(n-1)%6] + ' ' + first 3
   letters of nth lowercase NATO word"
    retrieval: the NATO alphabet    ONE. The six colours are spelled out and
               count for nothing; only the NATO words must be remembered. A
               rule with one named source and one written-out list is r = 1,
               not r = 2.
    operation: cycle the colours; first 3 letters; join       THREE
    -> r 1

  "Term n: for odd n, the ((n-1)/2 mod 7 + 1)th solfege syllable; for even n,
   the (n/2)th NATO word"
    retrieval: solfege syllables; the NATO alphabet           TWO
    operation: none. The parity branch and the two index expressions are one
               thing: ALTERNATING between the two sources in their natural
               order, and that is control. A rule written "for odd n ... for
               even n ..." is alternation SPELLED OUT in arithmetic. Do not
               count the spelling as work.
               It WOULD be an operation if the choice depended on a value
               rather than on parity, or if the two ran at different rates.
    -> r 2, t 0

  "Term n = rev(p_n[1..3]) + d_n[-1] + sum(len(p_i) + len(d_i)), with
   p = 3 Little Pigs materials, d = days of week"
    retrieval: the pigs' materials; the days of the week      TWO, even though
               p and d are each mentioned twice more
    operation: reverse; first 3 chars; last char; running total of lengths
    -> r 2

Quote each component as a span of the instruction. Do not invent components
that are not there.

Instruction: {p}

Then, separately: what human subject does this rule ACTUALLY draw on? Not what
it is dressed as. A rule that names three counters `root`, `stem` and `ear` and
adds them up draws on nothing: the words are decoration and the answer is
"none". A rule that indexes the NATO alphabet draws on that alphabet. Answer in
one or two words, or "none".

Answer in strict JSON, nothing else:
{{"components": [{{"text": "<span>", "type": "retrieval|operation|control"}}],
  "domain": "<one or two words, or none>"}}"""


## 5 · Measuring

In [ ]:
def type_once(text, temperature, why=None):
    """One call, returning distinct counts, or None if the reply is unusable.

    `why` collects the reason each failure happened. Without it every failure
    looked the same in the log, and an item that failed three times running
    was indistinguishable from one whose rule the call cannot handle: two
    items failed in both runs and there was no way to say what was wrong with
    them.
    """
    txt, note = ask(MEASURE.format(p=text), temperature=temperature,
                    max_out=MAX_OUT)
    if txt is None:
        if why is not None: why.append(f"no reply: {note}")
        return None
    # the shared parse_json returns (parsed, head), not the parse alone
    d, head = parse_json(txt)
    if d is None:
        if why is not None:
            why.append(f"reply was not JSON: {str(txt)[:70]!r}")
        return None
    n, seen = collections.Counter(), collections.defaultdict(set)
    for c in d.get("components", []):
        if not isinstance(c, dict): continue
        ty = str(c.get("type", "")).strip().lower()
        sp = str(c.get("text", "")).strip()
        if ty not in ("retrieval", "operation", "control"): continue
        if not (0 < len(sp) <= 120): continue
        # DISTINCT spans. A rule that names two retrievals and refers to each
        # twice more otherwise counts six. This catches literal repeats only;
        # an alias like p_n for "the pigs' materials" is what the worked
        # examples are for.
        key = " ".join(re.sub(r"[^a-z0-9 ]", " ", sp.lower()).split())
        if key in seen[ty]: continue
        seen[ty].add(key)
        n[ty] += 1
    dom = str(d.get("domain", "")).strip().lower()[:24] or "none"
    return dict(r=n["retrieval"], t=n["operation"], c=n["control"],
                domain="none" if dom in ("n/a", "-", "", "nothing") else dom)

def measure(item, pricing):
    """r and t for one item, read off the best description available.

    Three descriptions, in this order:

      p_star   the one that achieved K. Best, because then r_hat, t_hat and K
               all describe the SAME thing: one description, its cost, and what
               it is made of.
      bare     the shortest VERIFIED equivalent. The right kind of object, but
               it is the argmin of K on only 21 of 63 items, so usually it is
               not the one K refers to.
      seed     what the composer wrote. Always present.

    The fallbacks almost never fire. Every item that has been through the
    unquestionability pass has a p_star bar one. Where no equivalent verified,
    that pass falls back to the seed itself and records it as p_star, which is
    why 14 items have K_source "the seed rule" and are still measurable here.

    There is no minimisation across descriptions. Two of them can describe the
    same sequence at (1,3) and (3,1), so there is no true pair to minimise
    towards: minimising r+t assumes the axes are commensurable, minimising each
    separately gives a pair no description achieves, and averaging invents a
    point none occupies. Reading ONE description avoids the question, and
    reading the one that achieved K makes the answer agree with the cost.

    What is reported is therefore what the cheapest description is made of, not
    the item's complexity. Weaker, and true.
    """
    rec = (pricing or {}).get(item["id"]) or {}
    text, src = rec.get("p_star"), "p_star"
    if not text:
        text, src = item.get("bare"), "bare"
    if not text:
        text, src = item.get("seed"), "seed"
    if not text:
        return None, "no description of any kind on this item"

    why, got = [], []
    for _ in range(MEASURE_TRIES):
        if len(got) >= MEASURE_REPS: break
        g = type_once(text, MEASURE_TEMP, why)
        if g: got.append(g)
    if not got:
        return None, ("every typing call failed: "
                      + "; ".join(dict.fromkeys(why))[:200]
                      if why else "every typing call failed")
    return dict(
        r_hat=round(st.mean(g["r"] for g in got), 2),
        t_hat=round(st.mean(g["t"] for g in got), 2),
        r_hat_sd=round(st.pstdev([g["r"] for g in got]), 2),
        t_hat_sd=round(st.pstdev([g["t"] for g in got]), 2),
        r_hat_all=[g["r"] for g in got], t_hat_all=[g["t"] for g in got],
        n_typings=len(got),
        typed_from=src,
        # whether the description that achieved K was a verified equivalent or
        # the composer's own seed. Where it is the seed, nothing was ever
        # confirmed to reproduce the sequence.
        k_source=rec.get("K_source"),
        domain_hat=collections.Counter(g["domain"] for g in got)
                   .most_common(1)[0][0]), None


## 6 · Load

In [ ]:
BAT = json.load(open(find(BATTERY) or BATTERY))
ITEMS = BAT["items"]
try:
    PR = json.load(open(find(PRICING) or PRICING))
    PR = PR.get("items", PR)
except Exception as e:
    PR = {}
    log(f"  {PRICING} not read ({str(e)[:60]}), so nothing has a p_star")
log(f"  battery {len(ITEMS)} items, pricing {len(PR)} records")


## 7 · Run

In [ ]:
# MEASURE_ONLY names items to retype whatever they already hold. An item
# typed from the seed should be typed again once an equivalent exists, since
# r and t are meant to describe the same thing K prices.
todo = ([b for b in ITEMS if b["id"] in set(MEASURE_ONLY)] if MEASURE_ONLY
        else [b for b in ITEMS
              # THIS MACHINE'S field, not the plain one. The plain fields hold
              # whichever run measured last, so selecting on them made the
              # second machine's run skip every item the first had done.
              if MEASURE_ALL or b.get(f"r_hat_{PRICE_TAG}") is None])
have = sum(1 for b in todo if (PR.get(b["id"]) or {}).get("p_star"))
log(f"{len(ITEMS)} items, {len(todo)} to measure")
log(f"  {have} have a p_star, the description that achieved K, and that is "
    f"what gets read")
log(f"  {len(todo) - have} do not, and fall back to the bare, then the seed. "
    f"Every item has a seed, so")
log(f"  nothing is skipped; `typed_from` records which was used and the "
    f"report counts them.\n")

T0 = time.time()
done, skipped = [], []

def _one(b):
    rec, err = measure(b, PR)
    return b, rec, err

# parallel yields None where a worker raised, so those are dropped before
# unpacking rather than after
for _got in parallel(_one, todo, workers=MAX_WORKERS, label="typing"):
    if _got is None:
        continue
    b, rec, err = _got
    if rec is None:
        skipped.append((b["id"], err))
    else:
        b.update(rec)
        for _k in ("r_hat", "t_hat", "r_hat_sd", "t_hat_sd",
                   "r_hat_all", "t_hat_all", "n_typings", "typed_from"):
            if _k in rec:
                b[f"{_k}_{PRICE_TAG}"] = rec[_k]
        done.append(b)

log(f"\nmeasured {len(done)}, skipped {len(skipped)}")
for iid, why in skipped[:8]:
    log(f"   {iid}: {why}")


## 8 · What it says

In [ ]:
def _corr(a, b):
    n = len(a)
    if n < 3: return float("nan")
    ma, mb = sum(a)/n, sum(b)/n
    da = math.sqrt(sum((x-ma)**2 for x in a))
    db = math.sqrt(sum((y-mb)**2 for y in b))
    return sum((x-ma)*(y-mb) for x, y in zip(a, b))/(da*db) if da*db else 0.0

if done:
    firm = sum(1 for b in done if b["r_hat_sd"] == 0 and b["t_hat_sd"] == 0)
    log(f"{firm} of {len(done)} typed identically every time")
    log(f"mean spread: r {st.mean(b['r_hat_sd'] for b in done):.2f}, "
          f"t {st.mean(b['t_hat_sd'] for b in done):.2f}")

    src = collections.Counter(b.get("typed_from") for b in done)
    log(f"\nread from: " + ", ".join(f"{v} {k}" for k, v in src.most_common()))
    ks = collections.Counter(str(b.get("k_source")) for b in done)
    log(f"the description that achieved K was: "
        + ", ".join(f"{v} {k}" for k, v in ks.most_common()))
    log(f"  where that is the seed rule, nothing was ever confirmed to "
        f"reproduce the sequence, so r and t rest on the composer's wording.")

    log(f"\nasked against measured, the manipulation check:")
    log(f"   r  {_corr([b['want_retrieval'] for b in done], [b['r_hat'] for b in done]):+.2f}")
    log(f"   t  {_corr([b['want_transformation'] for b in done], [b['t_hat'] for b in done]):+.2f}")

    out = [b for b in done if b["r_hat"] > 2 or b["t_hat"] > 6]
    if out:
        log(f"\noutside the grid, not clipped, because clipping a "
              f"measurement to the design hides an item that needs more than "
              f"its cell allows:")
        for b in out:
            log(f"   {b['id']}  r {b['r_hat']} t {b['t_hat']}  "
                  f"asked r{b['want_retrieval']}t{b['want_transformation']}")

    var = sorted((b for b in done if b["r_hat_sd"] or b["t_hat_sd"]),
                 key=lambda b: -(b["r_hat_sd"] + b["t_hat_sd"]))[:8]
    if var:
        log(f"\nthe least settled typings:")
        for b in var:
            log(f"   {b['id']}  r {b['r_hat_all']}  t {b['t_hat_all']}")


## 9 · Save and push

In [ ]:
BAT["measured_by"] = BUILD
BAT["measured_on"] = datetime.datetime.now().isoformat(timespec="minutes")
with open(BATTERY, "w") as f:
    json.dump(BAT, f, indent=1)
log(f"\nwrote {BATTERY}: r_hat and t_hat on {len(done)} items")
log(f"measured_r and measured_t are untouched. They are the admission-time")
log(f"values and decided whether an item was accepted, so they are provenance")
log(f"rather than measurement.")

account(units=len(done), note=f"measured r,t on {len(done)} items")
state_save(f"{BUILD}: r_hat and t_hat on {len(done)} items")
